# A10b — Improved PoseNet 2D to Kinect 2D with Confidence, Smoothing, and GRU Tuning

This notebook keeps the same project paths and MLflow setup as A10, but focuses on three improvements:

1. Use PoseNet confidence scores as extra inputs
2. Smooth PoseNet trajectories before training
3. Run a narrow GRU-only hyperparameter search around the current champion

Main task:
- Input: PoseNet 2D joint positions + PoseNet confidence scores + PoseNet body-context features
- Target: Kinect 2D joint coordinates

This notebook:
- loads paired PoseNet and Kinect CSV files
- smooths PoseNet x/y trajectories
- computes body-normalized PoseNet features
- keeps Kinect targets in raw coordinates
- uses grouped k-fold cross validation
- trains GRU variants with early stopping
- reports cross-validated test MAE in cm
- saves and reloads the final champion model

### Imports and shared configuration

In [1]:
from pathlib import Path
import os
import random
import warnings
import copy
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import savgol_filter

from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import mlflow
import mlflow.pytorch
from mlflow.tracking import MlflowClient

from dotenv import load_dotenv
from os import getenv


warnings.filterwarnings("ignore")

# Reproducibility
RANDOM_SEED = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# Shared experiment settings
SEQ_LEN = 20
STRIDE = 1
N_SPLITS = 10
VALIDATION_SPLIT = 0.10

BATCH_SIZE = 32
EPOCHS = 100
PATIENCE = 15
LEARNING_RATE = 1e-3

LOSS_NAME = "huber"   # "huber" or "mse"
HUBER_DELTA = 1.0
DEFAULT_OPTIMIZER = "adam"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

POSENET_DIR = Path("../../data/raw/posenet_generated")
KINECT_DIR = Path("../../data/raw/kinect_good_preprocessed")

output_dir = Path("reporting/results")
output_dir.mkdir(parents=True, exist_ok=True)

artifact_dir = Path("reporting/artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

JOINTS = [
    "head",
    "left_shoulder",
    "left_elbow",
    "right_shoulder",
    "right_elbow",
    "left_hand",
    "right_hand",
    "left_hip",
    "right_hip",
    "left_knee",
    "right_knee",
    "left_foot",
    "right_foot",
]

FRAME_COL_CANDIDATES = ["FrameNo", "frame", "frame_no", "Frame", "frameno"]
SEQUENCE_COL_CANDIDATES = ["sequence_id", "SequenceId", "movement_id", "MovementId", "id", "ID"]

# PoseNet raw columns
PN_RAW_XY_FEATURES = [f"pn_{j}_{axis}" for j in JOINTS for axis in ["x", "y"]]
PN_SCORE_FEATURES = [f"pn_{j}_score" for j in JOINTS]

# PoseNet context features (added after smoothing)
PN_CONTEXT_FEATURES = [
    "pn_hip_center_x_raw",
    "pn_hip_center_y_raw",
    "pn_torso_len_raw",
]

# Final model inputs
MODEL_INPUT_FEATURES = PN_RAW_XY_FEATURES + PN_SCORE_FEATURES + PN_CONTEXT_FEATURES

# Kinect raw targets
TARGET_FEATURES = [f"k_{j}_{axis}" for j in JOINTS for axis in ["x", "y"]]

TARGET_UNIT = "meters"
CM_CONVERSION = 100.0 if TARGET_UNIT == "meters" else 1.0

NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

# Smoothing / preprocessing
USE_POSETNET_SMOOTHING = True
SMOOTHING_METHOD = "savgol"
SMOOTH_WINDOW = 7
SMOOTH_POLYORDER = 2

USE_CONFIDENCE_INPUTS = True
USE_BODY_NORMALIZED_POSE_INPUTS = True
USE_POSE_CONTEXT_INPUTS = True

print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("POSENET_DIR:", POSENET_DIR)
print("KINECT_DIR:", KINECT_DIR)

c:\Users\Kim\software engineering\year 4\LP3\4DT907\machine_learning_project_4dt907\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEVICE: cuda
GPU: NVIDIA GeForce RTX 2070
POSENET_DIR: ..\..\data\raw\posenet_generated
KINECT_DIR: ..\..\data\raw\kinect_good_preprocessed


### MLflow setup

In [2]:
load_dotenv("../../../../../env")

tracking_uri = getenv("MLFLOW_TRACKING_URI")
print("MLFLOW_TRACKING_URI =", tracking_uri)

if tracking_uri:
    mlflow.set_tracking_uri(tracking_uri)
else:
    raise RuntimeError("MLFLOW_TRACKING_URI not set. Fix env path/variable.")

EXPERIMENT_NAME = "A10b_pose_conf_smooth_gru"
mlflow.set_experiment(EXPERIMENT_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("MLflow experiment:", EXPERIMENT_NAME)

client = MlflowClient()

2026/04/21 20:25:42 INFO mlflow.tracking.fluent: Experiment with name 'A10b_pose_conf_smooth_gru' does not exist. Creating a new experiment.


MLFLOW_TRACKING_URI = https://172.27.62.170:5001/
MLflow tracking URI: https://172.27.62.170:5001/
MLflow experiment: A10b_pose_conf_smooth_gru


### Helper functions

In [5]:
def read_table_file(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".tsv":
        return pd.read_csv(path, sep="\t")
    if suffix == ".txt":
        try:
            df = pd.read_csv(path)
            if df.shape[1] > 1:
                return df
        except Exception:
            pass
        return pd.read_csv(path, delim_whitespace=True)
    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file type: {path}")


def find_frame_col(df: pd.DataFrame):
    lower_map = {c.lower(): c for c in df.columns}
    for col in FRAME_COL_CANDIDATES:
        if col.lower() in lower_map:
            return lower_map[col.lower()]
    return None


def find_sequence_col(df: pd.DataFrame):
    lower_map = {c.lower(): c for c in df.columns}
    for col in SEQUENCE_COL_CANDIDATES:
        if col.lower() in lower_map:
            return lower_map[col.lower()]
    return None


def standardize_joint_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    rename_map = {}
    for col in df.columns:
        normalized = col.strip().lower().replace(" ", "_").replace("-", "_")
        rename_map[col] = normalized
    return df.rename(columns=rename_map)


def extract_base_id(file_path: Path) -> str:
    name = file_path.stem
    name = name.replace("_kinect", "")
    name = name.replace("_posenet", "")
    name = name.replace("_pose", "")
    return name


def to_cm(value):
    return value * CM_CONVERSION

### Check files exist

In [6]:
posenet_files = sorted(POSENET_DIR.glob("*.csv"))
kinect_files = sorted(KINECT_DIR.glob("*.csv"))

print("PoseNet csv count:", len(posenet_files))
print("Kinect csv count:", len(kinect_files))

if len(posenet_files) == 0:
    raise FileNotFoundError(f"No PoseNet CSV files found in {POSENET_DIR}")
if len(kinect_files) == 0:
    raise FileNotFoundError(f"No Kinect CSV files found in {KINECT_DIR}")

PoseNet csv count: 181
Kinect csv count: 179


### Load and merge PoseNet + Kinect

In [7]:
def load_and_merge_posenet_kinect(posenet_dir: Path, kinect_dir: Path) -> pd.DataFrame:
    posenet_files = sorted(posenet_dir.glob("*.csv"))
    kinect_files = sorted(kinect_dir.glob("*.csv"))

    posenet_map = {extract_base_id(p): p for p in posenet_files}
    kinect_map = {extract_base_id(p): p for p in kinect_files}

    common_ids = sorted(set(posenet_map.keys()) & set(kinect_map.keys()))
    missing_in_posenet = sorted(set(kinect_map.keys()) - set(posenet_map.keys()))
    missing_in_kinect = sorted(set(posenet_map.keys()) - set(kinect_map.keys()))

    print("Matched sequences:", len(common_ids))
    print("Missing in PoseNet:", missing_in_posenet[:10])
    print("Missing in Kinect:", missing_in_kinect[:10])

    if not common_ids:
        raise ValueError("No matching PoseNet/Kinect files found.")

    merged_dfs = []

    for seq_id in common_ids:
        pn_path = posenet_map[seq_id]
        k_path = kinect_map[seq_id]

        pn_df = standardize_joint_column_names(read_table_file(pn_path))
        k_df = standardize_joint_column_names(read_table_file(k_path))

        pn_seq_col = find_sequence_col(pn_df)
        if pn_seq_col is None:
            pn_df["sequence_id"] = seq_id
        elif pn_seq_col != "sequence_id":
            pn_df = pn_df.rename(columns={pn_seq_col: "sequence_id"})

        pn_frame_col = find_frame_col(pn_df)
        if pn_frame_col is None:
            pn_df["frameno"] = np.arange(len(pn_df))
        elif pn_frame_col != "frameno":
            pn_df = pn_df.rename(columns={pn_frame_col: "frameno"})

        k_seq_col = find_sequence_col(k_df)
        if k_seq_col is None:
            k_df["sequence_id"] = seq_id
        elif k_seq_col != "sequence_id":
            k_df = k_df.rename(columns={k_seq_col: "sequence_id"})

        k_frame_col = find_frame_col(k_df)
        if k_frame_col is None:
            k_df["frameno"] = np.arange(len(k_df))
        elif k_frame_col != "frameno":
            k_df = k_df.rename(columns={k_frame_col: "frameno"})

        # Rename PoseNet columns
        pn_rename = {}
        for joint in JOINTS:
            for suffix in ["x", "y", "score"]:
                src = f"{joint}_{suffix}"
                dst = f"pn_{joint}_{suffix}"
                if src in pn_df.columns:
                    pn_rename[src] = dst
        pn_df = pn_df.rename(columns=pn_rename)

        # Ensure score columns exist
        for joint in JOINTS:
            score_col = f"pn_{joint}_score"
            if score_col not in pn_df.columns:
                pn_df[score_col] = 1.0

        # Rename Kinect columns
        k_rename = {}
        for joint in JOINTS:
            for suffix in ["x", "y", "z"]:
                src = f"{joint}_{suffix}"
                dst = f"k_{joint}_{suffix}"
                if src in k_df.columns:
                    k_rename[src] = dst
        k_df = k_df.rename(columns=k_rename)

        pn_keep = ["sequence_id", "frameno"] + PN_RAW_XY_FEATURES + PN_SCORE_FEATURES
        k_keep = ["sequence_id", "frameno"] + TARGET_FEATURES

        pn_keep = [c for c in pn_keep if c in pn_df.columns]
        k_keep = [c for c in k_keep if c in k_df.columns]

        pn_df = pn_df[pn_keep].copy()
        k_df = k_df[k_keep].copy()

        merged = pd.merge(pn_df, k_df, on=["sequence_id", "frameno"], how="inner")
        merged_dfs.append(merged)

    full_df = pd.concat(merged_dfs, ignore_index=True)
    full_df = full_df.sort_values(["sequence_id", "frameno"]).reset_index(drop=True)
    return full_df


df = load_and_merge_posenet_kinect(POSENET_DIR, KINECT_DIR)

print("Shape:", df.shape)
display(df.head())

missing_inputs = [c for c in PN_RAW_XY_FEATURES + PN_SCORE_FEATURES if c not in df.columns]
missing_targets = [c for c in TARGET_FEATURES if c not in df.columns]

print("Missing inputs:", missing_inputs)
print("Missing targets:", missing_targets)

assert not missing_inputs, "Missing PoseNet input columns."
assert not missing_targets, "Missing Kinect target columns."

Matched sequences: 179
Missing in PoseNet: []
Missing in Kinect: ['A60', 'posenet_generation_summary']
Shape: (24005, 67)


,sequence_id,frameno,pn_head_x,pn_head_y,pn_left_shoulder_x,pn_left_shoulder_y,pn_left_elbow_x,pn_left_elbow_y,pn_right_shoulder_x,pn_right_shoulder_y,...,k_right_hip_x,k_right_hip_y,k_left_knee_x,k_left_knee_y,k_right_knee_x,k_right_knee_y,k_left_foot_x,k_left_foot_y,k_right_foot_x,k_right_foot_y
0,A1,68,991.198604,260.810939,1050.207223,350.621848,1103.074503,287.405884,937.012138,336.460121,...,0.070990,0.050062,-0.12024,-0.31613,0.10949,-0.32352,-0.13160,-0.73613,0.13035,-0.74466
1,A1,69,991.070028,261.879878,1050.433990,350.473674,1102.743606,291.447918,937.913300,337.325910,...,0.071481,0.049983,-0.11961,-0.31636,0.11010,-0.32371,-0.13135,-0.73634,0.13078,-0.74484
2,A1,70,991.292346,261.123256,1050.139370,350.467175,1103.530488,288.592060,938.931798,338.126651,...,0.072057,0.049890,-0.11884,-0.31667,0.11070,-0.32399,-0.13110,-0.73665,0.13120,-0.74506
3,A1,71,991.415916,261.879359,1051.265302,349.772855,1103.656437,286.813590,936.743363,338.002401,...,0.072543,0.049759,-0.11805,-0.31707,0.11128,-0.32436,-0.13102,-0.73711,0.13164,-0.74529
4,A1,72,987.703924,266.026904,1049.559114,344.481013,1115.848482,328.638712,928.860345,333.770843,...,0.072987,0.049613,-0.11727,-0.31750,0.11181,-0.32480,-0.13096,-0.73750,0.13212,-0.74567


Missing inputs: []
Missing targets: []


### Smooth PoseNet x/y trajectories

In [8]:
def smooth_posenet_xy(
    df: pd.DataFrame,
    feature_cols,
    group_col="sequence_id",
    frame_col="frameno",
    method="savgol",
    window=7,
    polyorder=2,
):
    df = df.copy()

    def smooth_group(g):
        g = g.sort_values(frame_col).copy()

        for col in feature_cols:
            s = g[col].interpolate(method="linear", limit_direction="both").ffill().bfill()

            arr = s.to_numpy(dtype=np.float32)

            if len(arr) < 3:
                g[col] = arr
                continue

            if method == "savgol":
                win = min(window, len(arr))
                if win % 2 == 0:
                    win -= 1
                if win < 3:
                    g[col] = arr
                    continue

                effective_polyorder = min(polyorder, win - 1)
                try:
                    arr_sm = savgol_filter(arr, window_length=win, polyorder=effective_polyorder, mode="interp")
                except Exception:
                    arr_sm = pd.Series(arr).rolling(window=3, min_periods=1, center=True).mean().to_numpy()
            else:
                arr_sm = pd.Series(arr).rolling(window=3, min_periods=1, center=True).mean().to_numpy()

            g[col] = arr_sm

        return g

    return (
        df.groupby(group_col, group_keys=False)
        .apply(smooth_group)
        .reset_index(drop=True)
    )


if USE_POSETNET_SMOOTHING:
    df = smooth_posenet_xy(
        df=df,
        feature_cols=PN_RAW_XY_FEATURES,
        group_col="sequence_id",
        frame_col="frameno",
        method=SMOOTHING_METHOD,
        window=SMOOTH_WINDOW,
        polyorder=SMOOTH_POLYORDER,
    )
    print("Applied PoseNet smoothing.")
    display(df.head())

Applied PoseNet smoothing.


,sequence_id,frameno,pn_head_x,pn_head_y,pn_left_shoulder_x,pn_left_shoulder_y,pn_left_elbow_x,pn_left_elbow_y,pn_right_shoulder_x,pn_right_shoulder_y,...,k_right_hip_x,k_right_hip_y,k_left_knee_x,k_left_knee_y,k_right_knee_x,k_right_knee_y,k_left_foot_x,k_left_foot_y,k_right_foot_x,k_right_foot_y
0,A1,68,991.632385,260.625427,1050.421631,351.760529,1102.313110,284.640259,938.054871,337.062286,...,0.070990,0.050062,-0.12024,-0.31613,0.10949,-0.32352,-0.13160,-0.73613,0.13035,-0.74466
1,A1,69,991.053650,261.352081,1050.405884,349.818298,1102.250122,289.411804,937.395752,336.960999,...,0.071481,0.049983,-0.11961,-0.31636,0.11010,-0.32371,-0.13135,-0.73634,0.13078,-0.74484
2,A1,70,990.546509,262.176025,1050.267334,348.686768,1104.225220,294.982635,936.648438,337.005493,...,0.072057,0.049890,-0.11884,-0.31667,0.11070,-0.32399,-0.13110,-0.73665,0.13120,-0.74506
3,A1,71,990.111023,263.097198,1050.006104,348.365906,1108.238403,301.352783,935.812927,337.195801,...,0.072543,0.049759,-0.11805,-0.31707,0.11128,-0.32436,-0.13102,-0.73711,0.13164,-0.74529
4,A1,72,989.476318,264.366089,1049.243896,348.737549,1118.753784,311.964172,934.480774,337.466156,...,0.072987,0.049613,-0.11727,-0.31750,0.11181,-0.32480,-0.13096,-0.73750,0.13212,-0.74567


### Build PoseNet body-context features and body-normalized PoseNet inputs

In [9]:
def add_pose_context_and_normalized_inputs(df: pd.DataFrame, joint_names, eps=1e-6):
    df = df.copy()

    # Raw context from PoseNet
    df["pn_hip_center_x_raw"] = (df["pn_left_hip_x"] + df["pn_right_hip_x"]) / 2.0
    df["pn_hip_center_y_raw"] = (df["pn_left_hip_y"] + df["pn_right_hip_y"]) / 2.0

    shoulder_center_x = (df["pn_left_shoulder_x"] + df["pn_right_shoulder_x"]) / 2.0
    shoulder_center_y = (df["pn_left_shoulder_y"] + df["pn_right_shoulder_y"]) / 2.0

    torso_len = np.sqrt(
        (shoulder_center_x - df["pn_hip_center_x_raw"]) ** 2 +
        (shoulder_center_y - df["pn_hip_center_y_raw"]) ** 2
    )
    torso_len = np.where(torso_len < eps, 1.0, torso_len)
    df["pn_torso_len_raw"] = torso_len

    if USE_BODY_NORMALIZED_POSE_INPUTS:
        for joint in joint_names:
            x_col = f"pn_{joint}_x"
            y_col = f"pn_{joint}_y"

            df[x_col] = (df[x_col] - df["pn_hip_center_x_raw"]) / df["pn_torso_len_raw"]
            df[y_col] = (df[y_col] - df["pn_hip_center_y_raw"]) / df["pn_torso_len_raw"]

    return df


df = add_pose_context_and_normalized_inputs(df, JOINTS)
display(df.head())

,sequence_id,frameno,pn_head_x,pn_head_y,pn_left_shoulder_x,pn_left_shoulder_y,pn_left_elbow_x,pn_left_elbow_y,pn_right_shoulder_x,pn_right_shoulder_y,...,k_left_knee_y,k_right_knee_x,k_right_knee_y,k_left_foot_x,k_left_foot_y,k_right_foot_x,k_right_foot_y,pn_hip_center_x_raw,pn_hip_center_y_raw,pn_torso_len_raw
0,A1,68,0.019922,-1.409991,0.307971,-0.963457,0.562223,-1.292325,-0.242591,-1.035474,...,-0.31613,0.10949,-0.32352,-0.13160,-0.73613,0.13035,-0.74466,987.566406,548.396973,204.094650
1,A1,69,0.015496,-1.398064,0.303797,-0.968343,0.555629,-1.261765,-0.245145,-1.030797,...,-0.31636,0.11010,-0.32371,-0.13135,-0.73634,0.13078,-0.74484,987.863525,549.169922,205.868820
2,A1,70,0.011154,-1.388095,0.298699,-0.971562,0.558496,-1.230137,-0.248356,-1.027805,...,-0.31667,0.11070,-0.32399,-0.13110,-0.73665,0.13120,-0.74506,988.229980,550.472534,207.692215
3,A1,71,0.006898,-1.380025,0.292702,-0.973144,0.570571,-1.197479,-0.252199,-1.026445,...,-0.31707,0.11128,-0.32436,-0.13102,-0.73711,0.13164,-0.74529,988.665527,552.304810,209.566940
4,A1,72,0.000790,-1.370278,0.281919,-0.973419,0.608873,-1.146390,-0.257893,-1.026437,...,-0.31750,0.11181,-0.32480,-0.13096,-0.73750,0.13212,-0.74567,989.308350,555.684937,212.598419


### Interpolate remaining missing values

In [10]:
def interpolate_sequence_columns(df: pd.DataFrame, cols):
    df = df.copy()

    def interp_group(g):
        g = g.sort_values("frameno").copy()
        g[cols] = g[cols].interpolate(method="linear", limit_direction="both")
        g[cols] = g[cols].ffill().bfill()
        return g

    return (
        df.groupby("sequence_id", group_keys=False)
        .apply(interp_group)
        .reset_index(drop=True)
    )


interp_cols = MODEL_INPUT_FEATURES + TARGET_FEATURES
df = interpolate_sequence_columns(df, interp_cols)

display(df.head())

,sequence_id,frameno,pn_head_x,pn_head_y,pn_left_shoulder_x,pn_left_shoulder_y,pn_left_elbow_x,pn_left_elbow_y,pn_right_shoulder_x,pn_right_shoulder_y,...,k_left_knee_y,k_right_knee_x,k_right_knee_y,k_left_foot_x,k_left_foot_y,k_right_foot_x,k_right_foot_y,pn_hip_center_x_raw,pn_hip_center_y_raw,pn_torso_len_raw
0,A1,68,0.019922,-1.409991,0.307971,-0.963457,0.562223,-1.292325,-0.242591,-1.035474,...,-0.31613,0.10949,-0.32352,-0.13160,-0.73613,0.13035,-0.74466,987.566406,548.396973,204.094650
1,A1,69,0.015496,-1.398064,0.303797,-0.968343,0.555629,-1.261765,-0.245145,-1.030797,...,-0.31636,0.11010,-0.32371,-0.13135,-0.73634,0.13078,-0.74484,987.863525,549.169922,205.868820
2,A1,70,0.011154,-1.388095,0.298699,-0.971562,0.558496,-1.230137,-0.248356,-1.027805,...,-0.31667,0.11070,-0.32399,-0.13110,-0.73665,0.13120,-0.74506,988.229980,550.472534,207.692215
3,A1,71,0.006898,-1.380025,0.292702,-0.973144,0.570571,-1.197479,-0.252199,-1.026445,...,-0.31707,0.11128,-0.32436,-0.13102,-0.73711,0.13164,-0.74529,988.665527,552.304810,209.566940
4,A1,72,0.000790,-1.370278,0.281919,-0.973419,0.608873,-1.146390,-0.257893,-1.026437,...,-0.31750,0.11181,-0.32480,-0.13096,-0.73750,0.13212,-0.74567,989.308350,555.684937,212.598419


### Build fixed-length sequences

In [11]:
def build_sequences(
    df: pd.DataFrame,
    input_cols,
    target_cols,
    seq_len: int = 20,
    stride: int = 1,
):
    X_list = []
    y_list = []
    seq_ids = []

    clean_df = df.copy()

    for seq_id, group in clean_df.groupby("sequence_id"):
        group = group.sort_values("frameno").reset_index(drop=True)

        x_values = group[input_cols].to_numpy(dtype=np.float32)
        y_values = group[target_cols].to_numpy(dtype=np.float32)

        if len(group) < seq_len:
            continue

        for start in range(0, len(group) - seq_len + 1, stride):
            end = start + seq_len
            X_list.append(x_values[start:end])
            y_list.append(y_values[start:end])
            seq_ids.append(seq_id)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)
    seq_ids = np.array(seq_ids)

    return X, y, seq_ids


X, y, seq_ids = build_sequences(
    df=df,
    input_cols=MODEL_INPUT_FEATURES,
    target_cols=TARGET_FEATURES,
    seq_len=SEQ_LEN,
    stride=STRIDE,
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("seq_ids shape:", seq_ids.shape)
print("Unique sequences:", len(np.unique(seq_ids)))

X shape: (20604, 20, 42)
y shape: (20604, 20, 26)
seq_ids shape: (20604,)
Unique sequences: 179


### Dataset and scaling helpers

In [12]:
class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


INPUT_XY_INDICES = [MODEL_INPUT_FEATURES.index(c) for c in PN_RAW_XY_FEATURES]
INPUT_SCORE_INDICES = [MODEL_INPUT_FEATURES.index(c) for c in PN_SCORE_FEATURES if c in MODEL_INPUT_FEATURES]
INPUT_CONTEXT_INDICES = [MODEL_INPUT_FEATURES.index(c) for c in PN_CONTEXT_FEATURES if c in MODEL_INPUT_FEATURES]

def fit_input_scaler_on_training_sequences(X_train):
    scaler = StandardScaler()
    X_train_xy = X_train[:, :, INPUT_XY_INDICES].reshape(-1, len(INPUT_XY_INDICES))
    scaler.fit(X_train_xy)
    return scaler


def transform_input_sequences(X_data, scaler):
    X_out = X_data.copy()

    X_xy = X_out[:, :, INPUT_XY_INDICES].reshape(-1, len(INPUT_XY_INDICES))
    X_out[:, :, INPUT_XY_INDICES] = scaler.transform(X_xy).reshape(
        X_out.shape[0], X_out.shape[1], len(INPUT_XY_INDICES)
    )

    return X_out.astype(np.float32)


def fit_target_scaler_on_training_sequences(y_train):
    scaler = StandardScaler()
    y_2d = y_train.reshape(-1, y_train.shape[-1])
    scaler.fit(y_2d)
    return scaler


def transform_target_sequences(y_data, scaler):
    y_2d = y_data.reshape(-1, y_data.shape[-1])
    y_scaled = scaler.transform(y_2d).reshape(y_data.shape).astype(np.float32)
    return y_scaled


def inverse_transform_target_sequences(y_data, scaler):
    y_2d = y_data.reshape(-1, y_data.shape[-1])
    y_inv = scaler.inverse_transform(y_2d).reshape(y_data.shape).astype(np.float32)
    return y_inv

### GRU model

In [13]:
class GRUPoseNetToKinectRegressor(nn.Module):
    def __init__(self, n_features, n_targets, hidden_size=256, num_layers=2, dropout_rate=0.01, bidirectional=True):
        super().__init__()
        gru_dropout = dropout_rate if num_layers > 1 else 0.0

        self.gru = nn.GRU(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=gru_dropout,
            bidirectional=bidirectional,
        )

        direction_factor = 2 if bidirectional else 1
        self.dropout = nn.Dropout(dropout_rate) if dropout_rate > 0 else nn.Identity()
        self.output_layer = nn.Linear(hidden_size * direction_factor, n_targets)

    def forward(self, x):
        out, _ = self.gru(x)
        out = self.dropout(out)
        out = self.output_layer(out)
        return out


def build_model(config):
    return GRUPoseNetToKinectRegressor(
        n_features=len(MODEL_INPUT_FEATURES),
        n_targets=len(TARGET_FEATURES),
        hidden_size=config["hidden_size"],
        num_layers=config["num_layers"],
        dropout_rate=config["dropout_rate"],
        bidirectional=config["bidirectional"],
    )

### Training and evaluation helpers

In [14]:
def get_optimizer(model, optimizer_name="adam", lr=LEARNING_RATE):
    if optimizer_name.lower() == "adam":
        return torch.optim.Adam(model.parameters(), lr=lr)
    elif optimizer_name.lower() == "sgd":
        return torch.optim.SGD(model.parameters(), lr=lr)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")


def get_loss_function():
    if LOSS_NAME == "huber":
        return nn.HuberLoss(delta=HUBER_DELTA)
    return nn.MSELoss()


def evaluate_loss(model, data_loader, criterion):
    model.eval()
    total_loss = 0.0
    n_samples = 0

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.to(DEVICE, non_blocking=True)

            preds = model(X_batch)
            loss = criterion(preds, y_batch)

            batch_size = X_batch.size(0)
            total_loss += loss.item() * batch_size
            n_samples += batch_size

    return total_loss / max(n_samples, 1)


def predict_model(model, data_loader):
    model.eval()
    preds_all = []
    targets_all = []

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(DEVICE, non_blocking=True)
            preds = model(X_batch).cpu().numpy()
            preds_all.append(preds)
            targets_all.append(y_batch.numpy())

    y_pred = np.concatenate(preds_all, axis=0)
    y_true = np.concatenate(targets_all, axis=0)
    return y_true, y_pred


def mean_2d_distance(y_true, y_pred):
    n_joints = y_true.shape[-1] // 2
    distances = []

    for j in range(n_joints):
        x_idx = 2 * j
        y_idx = 2 * j + 1
        dx = y_pred[..., x_idx] - y_true[..., x_idx]
        dy = y_pred[..., y_idx] - y_true[..., y_idx]
        dist = np.sqrt(dx**2 + dy**2)
        distances.append(dist)

    distances = np.stack(distances, axis=-1)
    return float(np.mean(distances))


def per_joint_distance_cm(y_true, y_pred):
    result = {}
    for j, joint in enumerate(JOINTS):
        x_idx = 2 * j
        y_idx = 2 * j + 1
        dx = y_pred[..., x_idx] - y_true[..., x_idx]
        dy = y_pred[..., y_idx] - y_true[..., y_idx]
        dist = np.sqrt(dx**2 + dy**2)
        result[f"{joint}_dist_cm"] = float(np.mean(dist) * CM_CONVERSION)
    return result


def train_model(
    model,
    train_loader,
    val_loader,
    optimizer_name="adam",
    lr=LEARNING_RATE,
    epochs=EPOCHS,
    patience=PATIENCE,
):
    model = model.to(DEVICE)
    criterion = get_loss_function()
    optimizer = get_optimizer(model, optimizer_name=optimizer_name, lr=lr)

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_mae": [],
        "val_mae": [],
    }

    best_model_state = copy.deepcopy(model.state_dict())
    best_val_loss = float("inf")
    best_epoch = 0
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0
        total_train_mae = 0.0
        n_train = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.to(DEVICE, non_blocking=True)

            optimizer.zero_grad()
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()

            batch_size = X_batch.size(0)
            total_train_loss += loss.item() * batch_size
            total_train_mae += torch.mean(torch.abs(preds - y_batch)).item() * batch_size
            n_train += batch_size

        train_loss = total_train_loss / max(n_train, 1)
        train_mae = total_train_mae / max(n_train, 1)

        val_loss = evaluate_loss(model, val_loader, criterion)

        model.eval()
        total_val_mae = 0.0
        n_val = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(DEVICE, non_blocking=True)
                y_batch = y_batch.to(DEVICE, non_blocking=True)
                preds = model(X_batch)
                batch_size = X_batch.size(0)
                total_val_mae += torch.mean(torch.abs(preds - y_batch)).item() * batch_size
                n_val += batch_size

        val_mae = total_val_mae / max(n_val, 1)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_mae"].append(train_mae)
        history["val_mae"].append(val_mae)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    model.load_state_dict(best_model_state)
    return model, history, best_val_loss, best_epoch

### Narrow GRU-only sweep

In [15]:
experiment_configs = [
    {
        "model_family": "GRU",
        "model_variant": "BiGRU_192_2layers_drop001_lr1e3",
        "hidden_size": 192,
        "num_layers": 2,
        "dropout_rate": 0.01,
        "bidirectional": True,
        "learning_rate": 1e-3,
    },
    {
        "model_family": "GRU",
        "model_variant": "BiGRU_256_2layers_drop000_lr1e3",
        "hidden_size": 256,
        "num_layers": 2,
        "dropout_rate": 0.0,
        "bidirectional": True,
        "learning_rate": 1e-3,
    },
    {
        "model_family": "GRU",
        "model_variant": "BiGRU_256_2layers_drop001_lr1e3",
        "hidden_size": 256,
        "num_layers": 2,
        "dropout_rate": 0.01,
        "bidirectional": True,
        "learning_rate": 1e-3,
    },
    {
        "model_family": "GRU",
        "model_variant": "BiGRU_256_2layers_drop005_lr1e3",
        "hidden_size": 256,
        "num_layers": 2,
        "dropout_rate": 0.005,
        "bidirectional": True,
        "learning_rate": 1e-3,
    },
    {
        "model_family": "GRU",
        "model_variant": "BiGRU_256_2layers_drop020_lr1e3",
        "hidden_size": 256,
        "num_layers": 2,
        "dropout_rate": 0.02,
        "bidirectional": True,
        "learning_rate": 1e-3,
    },
    {
        "model_family": "GRU",
        "model_variant": "BiGRU_256_2layers_drop001_lr5e4",
        "hidden_size": 256,
        "num_layers": 2,
        "dropout_rate": 0.01,
        "bidirectional": True,
        "learning_rate": 5e-4,
    },
    {
        "model_family": "GRU",
        "model_variant": "BiGRU_320_2layers_drop001_lr1e3",
        "hidden_size": 320,
        "num_layers": 2,
        "dropout_rate": 0.01,
        "bidirectional": True,
        "learning_rate": 1e-3,
    },
]

experiment_configs

[{'model_family': 'GRU',
  'model_variant': 'BiGRU_192_2layers_drop001_lr1e3',
  'hidden_size': 192,
  'num_layers': 2,
  'dropout_rate': 0.01,
  'bidirectional': True,
  'learning_rate': 0.001},
 {'model_family': 'GRU',
  'model_variant': 'BiGRU_256_2layers_drop000_lr1e3',
  'hidden_size': 256,
  'num_layers': 2,
  'dropout_rate': 0.0,
  'bidirectional': True,
  'learning_rate': 0.001},
 {'model_family': 'GRU',
  'model_variant': 'BiGRU_256_2layers_drop001_lr1e3',
  'hidden_size': 256,
  'num_layers': 2,
  'dropout_rate': 0.01,
  'bidirectional': True,
  'learning_rate': 0.001},
 {'model_family': 'GRU',
  'model_variant': 'BiGRU_256_2layers_drop005_lr1e3',
  'hidden_size': 256,
  'num_layers': 2,
  'dropout_rate': 0.005,
  'bidirectional': True,
  'learning_rate': 0.001},
 {'model_family': 'GRU',
  'model_variant': 'BiGRU_256_2layers_drop020_lr1e3',
  'hidden_size': 256,
  'num_layers': 2,
  'dropout_rate': 0.02,
  'bidirectional': True,
  'learning_rate': 0.001},
 {'model_family': 'G

### Cross-validation runner

In [16]:
def run_cv_experiment(X, y, seq_ids, config):
    gkf = GroupKFold(n_splits=N_SPLITS)

    fold_results = []
    fold_histories = []

    with mlflow.start_run(run_name=config["model_variant"], nested=False):
        mlflow.log_param("model_family", config["model_family"])
        mlflow.log_param("model_variant", config["model_variant"])
        mlflow.log_param("sequence_length", SEQ_LEN)
        mlflow.log_param("stride", STRIDE)
        mlflow.log_param("n_splits", N_SPLITS)
        mlflow.log_param("batch_size", BATCH_SIZE)
        mlflow.log_param("epochs", EPOCHS)
        mlflow.log_param("patience", PATIENCE)
        mlflow.log_param("optimizer", DEFAULT_OPTIMIZER)
        mlflow.log_param("loss", LOSS_NAME)
        mlflow.log_param("huber_delta", HUBER_DELTA)
        mlflow.log_param("target_unit", TARGET_UNIT)
        mlflow.log_param("device", str(DEVICE))
        mlflow.log_param("pose_smoothing", USE_POSETNET_SMOOTHING)
        mlflow.log_param("pose_smoothing_method", SMOOTHING_METHOD)
        mlflow.log_param("use_confidence_inputs", USE_CONFIDENCE_INPUTS)
        mlflow.log_param("use_body_normalized_pose_inputs", USE_BODY_NORMALIZED_POSE_INPUTS)
        mlflow.log_param("use_pose_context_inputs", USE_POSE_CONTEXT_INPUTS)

        for key, value in config.items():
            if key not in ["model_family", "model_variant"]:
                mlflow.log_param(key, str(value))

        for fold_idx, (train_val_idx, test_idx) in enumerate(gkf.split(X, y, groups=seq_ids), start=1):
            X_train_val = X[train_val_idx]
            y_train_val = y[train_val_idx]
            seq_train_val = seq_ids[train_val_idx]

            X_test = X[test_idx]
            y_test = y[test_idx]

            unique_train_sequences = np.unique(seq_train_val)
            train_seq_ids, val_seq_ids = train_test_split(
                unique_train_sequences,
                test_size=VALIDATION_SPLIT,
                random_state=RANDOM_SEED + fold_idx,
            )

            train_mask = np.isin(seq_train_val, train_seq_ids)
            val_mask = np.isin(seq_train_val, val_seq_ids)

            X_train = X_train_val[train_mask]
            y_train = y_train_val[train_mask]
            X_val = X_train_val[val_mask]
            y_val = y_train_val[val_mask]

            x_scaler = fit_input_scaler_on_training_sequences(X_train)
            y_scaler = fit_target_scaler_on_training_sequences(y_train)

            X_train_scaled = transform_input_sequences(X_train, x_scaler)
            X_val_scaled = transform_input_sequences(X_val, x_scaler)
            X_test_scaled = transform_input_sequences(X_test, x_scaler)

            y_train_scaled = transform_target_sequences(y_train, y_scaler)
            y_val_scaled = transform_target_sequences(y_val, y_scaler)
            y_test_scaled = transform_target_sequences(y_test, y_scaler)

            train_loader = DataLoader(
                SequenceDataset(X_train_scaled, y_train_scaled),
                batch_size=BATCH_SIZE,
                shuffle=True,
                num_workers=NUM_WORKERS,
                pin_memory=PIN_MEMORY,
            )
            val_loader = DataLoader(
                SequenceDataset(X_val_scaled, y_val_scaled),
                batch_size=BATCH_SIZE,
                shuffle=False,
                num_workers=NUM_WORKERS,
                pin_memory=PIN_MEMORY,
            )
            test_loader = DataLoader(
                SequenceDataset(X_test_scaled, y_test_scaled),
                batch_size=BATCH_SIZE,
                shuffle=False,
                num_workers=NUM_WORKERS,
                pin_memory=PIN_MEMORY,
            )

            model = build_model(config)
            lr = config.get("learning_rate", LEARNING_RATE)

            trained_model, history, best_val_loss, best_epoch = train_model(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
                optimizer_name=DEFAULT_OPTIMIZER,
                lr=lr,
                epochs=EPOCHS,
                patience=PATIENCE,
            )

            y_true_scaled, y_pred_scaled = predict_model(trained_model, test_loader)

            y_true = inverse_transform_target_sequences(y_true_scaled, y_scaler)
            y_pred = inverse_transform_target_sequences(y_pred_scaled, y_scaler)

            test_mae = mean_absolute_error(
                y_true.reshape(-1, y_true.shape[-1]),
                y_pred.reshape(-1, y_pred.shape[-1]),
            )
            test_mae_cm = to_cm(test_mae)

            avg_2d_dist = mean_2d_distance(y_true, y_pred)
            avg_2d_dist_cm = to_cm(avg_2d_dist)

            fold_result = {
                "fold": fold_idx,
                "model_family": config["model_family"],
                "model_variant": config["model_variant"],
                "test_mae": float(test_mae),
                "test_mae_cm": float(test_mae_cm),
                "avg_2d_dist": float(avg_2d_dist),
                "avg_2d_dist_cm": float(avg_2d_dist_cm),
                "best_val_loss": float(best_val_loss),
                "best_epoch": int(best_epoch),
                "n_train_sequences": int(len(X_train)),
                "n_val_sequences": int(len(X_val)),
                "n_test_sequences": int(len(X_test)),
            }

            fold_result.update(per_joint_distance_cm(y_true, y_pred))
            fold_results.append(fold_result)
            fold_histories.append(history)

            mlflow.log_metric(f"fold_{fold_idx}_test_mae_cm", float(test_mae_cm))
            mlflow.log_metric(f"fold_{fold_idx}_avg_2d_dist_cm", float(avg_2d_dist_cm))
            mlflow.log_metric(f"fold_{fold_idx}_best_val_loss", float(best_val_loss))
            mlflow.log_metric(f"fold_{fold_idx}_best_epoch", int(best_epoch))

        fold_df = pd.DataFrame(fold_results)

        mlflow.log_metric("mean_test_mae", float(fold_df["test_mae"].mean()))
        mlflow.log_metric("std_test_mae", float(fold_df["test_mae"].std(ddof=1)))
        mlflow.log_metric("mean_test_mae_cm", float(fold_df["test_mae_cm"].mean()))
        mlflow.log_metric("std_test_mae_cm", float(fold_df["test_mae_cm"].std(ddof=1)))
        mlflow.log_metric("mean_avg_2d_dist_cm", float(fold_df["avg_2d_dist_cm"].mean()))
        mlflow.log_metric("std_avg_2d_dist_cm", float(fold_df["avg_2d_dist_cm"].std(ddof=1)))

    return fold_results, fold_histories

### Run the GRU tuning round

In [ ]:
all_fold_results = []
all_histories = {}
failed_configs = []

for config in experiment_configs:
    print(f"Running {config['model_variant']} ...")
    try:
        fold_results, fold_histories = run_cv_experiment(X, y, seq_ids, config)
        all_fold_results.extend(fold_results)
        all_histories[config["model_variant"]] = fold_histories
    except Exception as e:
        print(f"FAILED: {config['model_variant']} -> {e}")
        failed_configs.append((config["model_variant"], str(e)))

results_folds = pd.DataFrame(all_fold_results)
display(results_folds.head())

print("\nFailed configs:")
for name, err in failed_configs:
    print(f"- {name}: {err}")

Running BiGRU_192_2layers_drop001_lr1e3 ...
🏃 View run BiGRU_192_2layers_drop001_lr1e3 at: https://172.27.62.170:5001/#/experiments/27/runs/5a956168c9c74be0b9f44337ef873153
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/27
Running BiGRU_256_2layers_drop000_lr1e3 ...
🏃 View run BiGRU_256_2layers_drop000_lr1e3 at: https://172.27.62.170:5001/#/experiments/27/runs/41b93ed347f2448b8d55a09024b169fb
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/27
Running BiGRU_256_2layers_drop001_lr1e3 ...
🏃 View run BiGRU_256_2layers_drop001_lr1e3 at: https://172.27.62.170:5001/#/experiments/27/runs/5ecd9e86822c470cb529da8799b88532
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/27
Running BiGRU_256_2layers_drop005_lr1e3 ...
🏃 View run BiGRU_256_2layers_drop005_lr1e3 at: https://172.27.62.170:5001/#/experiments/27/runs/4663d1c749ca4ffd8712fa67d46a709e
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/27
Running BiGRU_256_2layers_drop020_lr1e3 ...


### Summarize results and choose champion

In [ ]:
results_summary = (
    results_folds
    .groupby(["model_family", "model_variant"], as_index=False)
    .agg(
        mean_test_mae=("test_mae", "mean"),
        std_test_mae=("test_mae", "std"),
        mean_test_mae_cm=("test_mae_cm", "mean"),
        std_test_mae_cm=("test_mae_cm", "std"),
        mean_avg_2d_dist_cm=("avg_2d_dist_cm", "mean"),
        std_avg_2d_dist_cm=("avg_2d_dist_cm", "std"),
        mean_best_val_loss=("best_val_loss", "mean"),
        mean_best_epoch=("best_epoch", "mean"),
    )
    .sort_values(["mean_avg_2d_dist_cm", "mean_test_mae_cm"], ascending=True)
    .reset_index(drop=True)
)

display(results_summary)

champion_row = results_summary.iloc[0]
champion_family = champion_row["model_family"]
champion_variant = champion_row["model_variant"]

print("Champion family:", champion_family)
print("Champion variant:", champion_variant)
display(champion_row)

### Plot training curves for the champion

In [ ]:
def plot_history(history, title):
    plt.figure(figsize=(8, 4))
    plt.plot(history["train_loss"], label="train_loss")
    plt.plot(history["val_loss"], label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.show()


champion_histories = all_histories[champion_variant]
plot_history(champion_histories[0], f"{champion_variant} - Fold 1")

### Save results

In [ ]:
fold_results_path = output_dir / "A10b_pose_conf_smooth_gru_folds.csv"
summary_results_path = output_dir / "A10b_pose_conf_smooth_gru_summary.csv"

results_folds.to_csv(fold_results_path, index=False)
results_summary.to_csv(summary_results_path, index=False)

print("Saved fold results to:", fold_results_path)
print("Saved summary results to:", summary_results_path)

### Retrain champion on almost all data

In [ ]:
champion_config = next(cfg for cfg in experiment_configs if cfg["model_variant"] == champion_variant)

unique_seq_ids = np.unique(seq_ids)
train_seq_ids, val_seq_ids = train_test_split(
    unique_seq_ids,
    test_size=VALIDATION_SPLIT,
    random_state=RANDOM_SEED,
)

train_mask = np.isin(seq_ids, train_seq_ids)
val_mask = np.isin(seq_ids, val_seq_ids)

X_train_final = X[train_mask]
y_train_final = y[train_mask]
X_val_final = X[val_mask]
y_val_final = y[val_mask]

x_scaler_final = fit_input_scaler_on_training_sequences(X_train_final)
y_scaler_final = fit_target_scaler_on_training_sequences(y_train_final)

X_train_final_scaled = transform_input_sequences(X_train_final, x_scaler_final)
X_val_final_scaled = transform_input_sequences(X_val_final, x_scaler_final)

y_train_final_scaled = transform_target_sequences(y_train_final, y_scaler_final)
y_val_final_scaled = transform_target_sequences(y_val_final, y_scaler_final)

train_loader_final = DataLoader(
    SequenceDataset(X_train_final_scaled, y_train_final_scaled),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

val_loader_final = DataLoader(
    SequenceDataset(X_val_final_scaled, y_val_final_scaled),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

final_model = build_model(champion_config)

print("Final train sequences:", len(X_train_final))
print("Final val sequences:", len(X_val_final))
print("Final model:", champion_variant)

### Train and save champion model

In [ ]:
final_lr = champion_config.get("learning_rate", LEARNING_RATE)

with mlflow.start_run(run_name=f"{champion_variant}_final", nested=False):
    mlflow.log_param("final_model_family", champion_family)
    mlflow.log_param("final_model_variant", champion_variant)
    mlflow.log_param("final_learning_rate", final_lr)

    final_model, final_history, final_best_val_loss, final_best_epoch = train_model(
        model=final_model,
        train_loader=train_loader_final,
        val_loader=val_loader_final,
        optimizer_name=DEFAULT_OPTIMIZER,
        lr=final_lr,
        epochs=EPOCHS,
        patience=PATIENCE,
    )

    mlflow.log_metric("final_best_val_loss", float(final_best_val_loss))
    mlflow.log_metric("final_best_epoch", int(final_best_epoch))

    final_model_path = artifact_dir / "A10b_champion_model.pt"
    final_x_scaler_path = artifact_dir / "A10b_x_scaler.npz"
    final_y_scaler_path = artifact_dir / "A10b_y_scaler.npz"
    final_metadata_path = artifact_dir / "A10b_champion_metadata.json"

    torch.save(final_model.state_dict(), final_model_path)

    np.savez(final_x_scaler_path, mean=x_scaler_final.mean_, scale=x_scaler_final.scale_)
    np.savez(final_y_scaler_path, mean=y_scaler_final.mean_, scale=y_scaler_final.scale_)

    metadata = {
        "model_family": champion_family,
        "model_variant": champion_variant,
        "sequence_length": SEQ_LEN,
        "input_features": MODEL_INPUT_FEATURES,
        "target_features": TARGET_FEATURES,
        "target_unit": TARGET_UNIT,
        "mean_cv_test_mae": float(champion_row["mean_test_mae"]),
        "mean_cv_test_mae_cm": float(champion_row["mean_test_mae_cm"]),
        "std_cv_test_mae_cm": float(champion_row["std_test_mae_cm"]),
        "mean_cv_avg_2d_dist_cm": float(champion_row["mean_avg_2d_dist_cm"]),
        "std_cv_avg_2d_dist_cm": float(champion_row["std_avg_2d_dist_cm"]),
        "best_epoch_final": int(final_best_epoch),
        "device": str(DEVICE),
        "loss_name": LOSS_NAME,
        "config": {k: list(v) if isinstance(v, tuple) else v for k, v in champion_config.items()},
    }

    with open(final_metadata_path, "w") as f:
        json.dump(metadata, f, indent=2)

    mlflow.log_artifact(str(final_model_path))
    mlflow.log_artifact(str(final_x_scaler_path))
    mlflow.log_artifact(str(final_y_scaler_path))
    mlflow.log_artifact(str(final_metadata_path))

print("Saved champion model to:", final_model_path)
print("Saved X scaler to:", final_x_scaler_path)
print("Saved Y scaler to:", final_y_scaler_path)
print("Saved metadata to:", final_metadata_path)

### Reload the saved champion model

In [ ]:
def load_saved_scaler(npz_path):
    data = np.load(npz_path)
    scaler = StandardScaler()
    scaler.mean_ = data["mean"]
    scaler.scale_ = data["scale"]
    scaler.var_ = scaler.scale_ ** 2
    scaler.n_features_in_ = len(scaler.mean_)
    return scaler


with open(final_metadata_path, "r") as f:
    loaded_meta = json.load(f)

loaded_config = loaded_meta["config"]

loaded_model = build_model(loaded_config)
loaded_model.load_state_dict(torch.load(final_model_path, map_location=DEVICE))
loaded_model.to(DEVICE)
loaded_model.eval()

loaded_x_scaler = load_saved_scaler(final_x_scaler_path)
loaded_y_scaler = load_saved_scaler(final_y_scaler_path)

print("Loaded model:", loaded_meta["model_variant"])

### Quick prediction check

In [ ]:
X_sample = X[:5]
y_sample = y[:5]

X_sample_scaled = transform_input_sequences(X_sample, loaded_x_scaler)
y_sample_scaled = transform_target_sequences(y_sample, loaded_y_scaler)

sample_ds = SequenceDataset(X_sample_scaled, y_sample_scaled)
sample_loader = DataLoader(sample_ds, batch_size=5, shuffle=False)

y_true_scaled, y_pred_scaled = predict_model(loaded_model, sample_loader)

y_true_sample = inverse_transform_target_sequences(y_true_scaled, loaded_y_scaler)
y_pred_sample = inverse_transform_target_sequences(y_pred_scaled, loaded_y_scaler)

sample_mae = mean_absolute_error(
    y_true_sample.reshape(-1, y_true_sample.shape[-1]),
    y_pred_sample.reshape(-1, y_pred_sample.shape[-1]),
)

sample_avg_2d_dist = mean_2d_distance(y_true_sample, y_pred_sample)

print("Sample MAE:", sample_mae)
print("Sample MAE (cm):", to_cm(sample_mae))
print("Sample average 2D distance (cm):", to_cm(sample_avg_2d_dist))

### Plots for analytical purposes

In [ ]:
# Presentation plots: model comparison, fold stability, champion training curves, and per-joint errors

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Use tuned results if they exist, otherwise fall back to the regular summary/results
summary_df = tuned_results_summary.copy() if "tuned_results_summary" in globals() else results_summary.copy()
folds_df = tuned_results_folds.copy() if "tuned_results_folds" in globals() else results_folds.copy()

chosen_variant = tuned_champion_variant if "tuned_champion_variant" in globals() else champion_variant
chosen_histories = tuned_histories[chosen_variant] if "tuned_histories" in globals() else all_histories[chosen_variant]

print("Using champion variant:", chosen_variant)

# -----------------------------
# 1) Mean test MAE (cm) by model
# -----------------------------
plot_df = summary_df.sort_values("mean_test_mae_cm", ascending=True).copy()

plt.figure(figsize=(10, 6))
plt.barh(plot_df["model_variant"], plot_df["mean_test_mae_cm"])
plt.xlabel("Mean test MAE (cm)")
plt.ylabel("Model variant")
plt.title("Model comparison: cross-validated mean test MAE (cm)")
plt.tight_layout()
plt.show()

# -----------------------------
# 2) Fold-by-fold stability for each model
# -----------------------------
variants_in_order = (
    folds_df.groupby("model_variant")["test_mae_cm"]
    .mean()
    .sort_values()
    .index
    .tolist()
)

box_data = [
    folds_df.loc[folds_df["model_variant"] == variant, "test_mae_cm"].values
    for variant in variants_in_order
]

plt.figure(figsize=(12, 6))
plt.boxplot(box_data, labels=variants_in_order, vert=True)
plt.ylabel("Fold test MAE (cm)")
plt.xlabel("Model variant")
plt.title("Fold-to-fold variation in test MAE (cm)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# -----------------------------
# 3) Champion learning curves (mean across folds)
# -----------------------------
max_len = max(len(h["train_loss"]) for h in chosen_histories)

def pad_with_nan(values, target_len):
    arr = np.array(values, dtype=float)
    if len(arr) < target_len:
        arr = np.concatenate([arr, np.full(target_len - len(arr), np.nan)])
    return arr

train_loss_mat = np.vstack([pad_with_nan(h["train_loss"], max_len) for h in chosen_histories])
val_loss_mat = np.vstack([pad_with_nan(h["val_loss"], max_len) for h in chosen_histories])

mean_train_loss = np.nanmean(train_loss_mat, axis=0)
mean_val_loss = np.nanmean(val_loss_mat, axis=0)

plt.figure(figsize=(9, 5))
plt.plot(range(1, max_len + 1), mean_train_loss, label="mean train loss")
plt.plot(range(1, max_len + 1), mean_val_loss, label="mean val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"Champion learning curve: {chosen_variant}")
plt.legend()
plt.tight_layout()
plt.show()

# -----------------------------
# 4) Per-joint error for the champion
# -----------------------------
champion_fold_df = folds_df[folds_df["model_variant"] == chosen_variant].copy()
joint_cols = [f"{joint}_dist_cm" for joint in JOINTS if f"{joint}_dist_cm" in champion_fold_df.columns]

joint_mean = champion_fold_df[joint_cols].mean().sort_values()
joint_std = champion_fold_df[joint_cols].std().reindex(joint_mean.index)

pretty_joint_names = [c.replace("_dist_cm", "") for c in joint_mean.index]

plt.figure(figsize=(10, 6))
plt.barh(pretty_joint_names, joint_mean.values, xerr=joint_std.values)
plt.xlabel("Mean joint distance error (cm)")
plt.ylabel("Joint")
plt.title(f"Champion per-joint error: {chosen_variant}")
plt.tight_layout()
plt.show()

# -----------------------------
# 5) Summary table for presentation
# -----------------------------
presentation_table = summary_df[
    ["model_family", "model_variant", "mean_test_mae_cm", "std_test_mae_cm", "mean_avg_2d_dist_cm", "std_avg_2d_dist_cm", "mean_best_epoch"]
].sort_values("mean_test_mae_cm", ascending=True).reset_index(drop=True)

display(presentation_table)